# 🌍 GeoAI Bootcamp — Kaggle ETL Pipeline
## Production-Grade Data Engineering for Geospatial AI

---
# ⚙️ SECTION 1 — Environment Setup

Welcome to the GeoAI ETL Pipeline! 🚀

Before we can extract data or run artificial intelligence models, we need to prepare our Kaggle environment. Kaggle provides a powerful computer (with a free GPU!), but it comes as a "blank slate."

**In this section, we will:**
1. **Install Dependencies:** Download all the specialized Python libraries we need (like `geopandas` for mapping, `unsloth` for fast AI, and `psycopg2` for database connections).
2. **Configure Networking:** Apply a special network patch to force Kaggle to use IPv4, which is required to successfully talk to the US Census Bureau servers.

> [!IMPORTANT]
> **Run cells 1.1 → 1.3 ONCE at the very beginning of each Kaggle session.** 
> If your session times out or you restart the kernel, you must come back and run these cells again before continuing!


### 📦 1.1 — Install All Dependencies

To build a modern GeoAI pipeline, we need tools from several different fields of data science. This single cell installs all the external packages we need so we don't have to install them later.

**What we are installing:**
* **Spatial Data Tools:** `geopandas`, `shapely`, `fiona` (Allows us to read and manipulate complex geographic boundaries and coordinates).
* **Database Connectors:** `sqlalchemy`, `psycopg2-binary`, `geoalchemy2` (Allows Python to securely push data and geometries into our local PostGIS database).
* **AI & Machine Learning:** `unsloth`, `sentence-transformers`, `nltk` (The engines that run our Large Language Models and generate our semantic knowledge graphs).

*(Note: The `-q` flag stands for "quiet," which prevents Kaggle from flooding the screen with hundreds of lines of installation text!)*


In [5]:
# @title 📦 1.1 — Install All Dependencies (Run Once Per Session)
!pip install -q pandas numpy scikit-learn geopandas shapely pyproj fiona geoalchemy2
!pip install -q sentence-transformers rake-nltk nltk
!pip install -q pyarrow fastparquet sqlalchemy psycopg2-binary python-dotenv kaggle
!pip install -q unsloth unsloth_zoo
!pip install -q unidecode          
!pip install -q kagglehub          

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    import unsloth
    print(f"✅ Unsloth version: {unsloth.__version__}")
except ImportError:
    print("❌ Unsloth failed to install!")

print("✅ All dependencies installed.")


✅ Unsloth version: 2026.8.22
✅ All dependencies installed.


### 🖥️ 1.2 — GPU / CUDA Validation

Large Language Models (LLMs) require massive amounts of mathematical processing. If we run our Llama 3.2 model on a standard CPU, it could take days to process the reviews. By running it on a GPU (Graphics Processing Unit), we can process the data exponentially faster.

This cell runs a hardware diagnostic to communicate with the Kaggle servers and verify that we successfully secured a GPU (usually a Tesla T4). 

> [!TIP]
> If this cell warns you that **"No GPU found,"** you need to turn it on!
> Go to the top right of your Kaggle notebook, click **Session Options** (the three dots or the hardware icon), select **Accelerator**, and choose **GPU T4 x2**. You will need to restart your session after doing this.


In [ ]:
# @title 🖥️ 1.2 — GPU / CUDA Validation
import torch

print("=" * 55)
print("GPU ENVIRONMENT CHECK")
print("=" * 55)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory/1024**3:.1f} GB")
    print(f"  CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")
else:
    print("  ⚠️ No GPU found — LLM will be extremely slow on CPU!")
print("=" * 55)


### 🌐 1.3 — Pre-Flight Connectivity & Environment Test

In Data Engineering, you should never start a 10-hour processing job without verifying your infrastructure first! This "Pre-Flight" cell runs three critical health checks:

1. **The IPv4 Routing Fix (CRITICAL):** The US Census Bureau's servers have a strict firewall that sometimes blocks modern IPv6 connections coming from Linux machines (like Kaggle). This code applies a hidden "patch" to Python's networking engine, forcing it to use the older, universally accepted IPv4 protocol. Without this, your Census downloads will fail!
2. **Disk Space Check:** Processing 180,000 reviews generates massive files. We check if Kaggle has at least 5GB of free hard drive space.
3. **VRAM Check:** Verifies that your GPU has enough free video memory to load the AI models into memory without crashing.


In [ ]:
# @title 🌐 1.3 — Pre-Flight Connectivity & Environment Test
# ⭐ Run this BEFORE starting any ETL. It validates:
#   - Census Bureau API is reachable (with IPv4 fix applied)
#   - Disk space is sufficient
#   - GPU memory is available

import os, socket, requests, shutil

# ── IPv4 Routing Fix ──────────────────────────────────────────────────────────
# Kaggle's Linux environment routes some requests over IPv6, which the Census
# Bureau firewall rejects. This hook forces all DNS lookups to use IPv4.
_orig_getaddrinfo = socket.getaddrinfo
def _ipv4_only(host, port, family=0, type=0, proto=0, flags=0):
    return _orig_getaddrinfo(host, port, socket.AF_INET, type, proto, flags)
socket.getaddrinfo = _ipv4_only

print("=" * 55)
print("PRE-FLIGHT CHECKS")
print("=" * 55)

# 1. Census API reachability
try:
    r = requests.get("https://api.census.gov/data.json", timeout=10)
    status = "✅ REACHABLE" if r.status_code == 200 else f"⚠️ HTTP {r.status_code}"
except Exception as e:
    status = f"❌ UNREACHABLE ({e})"
print(f"  Census API  : {status}")

# 2. Disk space
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"  Disk Space  : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
if free/1024**3 < 5:
    print("  ⚠️  WARNING: Less than 5 GB free — pipeline may run out of space!")

# 3. GPU memory
import torch
if torch.cuda.is_available():
    free_vram = torch.cuda.mem_get_info()[0] / 1024**3
    total_vram = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"  GPU VRAM    : {free_vram:.1f} GB free / {total_vram:.1f} GB total")
else:
    print("  GPU VRAM    : No GPU detected")

print("=" * 55)
print("✅ Pre-flight checks complete. Safe to begin ETL pipeline.")


---
# 🎛️ SECTION 2 — Global Configuration

Welcome to the "Control Room" of your ETL pipeline! 🎛️

In messy codebases, variables like API keys, target cities, and batch sizes are scattered across 20 different cells. This makes the code almost impossible to maintain or scale. 

**Best Practice: The Single Source of Truth**
We have gathered every single customizable parameter into this one section. If you want to change the pipeline to analyze a different city, use a different AI model, or process the data from a different year, you **only** need to edit the code here.

> [!TIP]
> Before you run the rest of the notebook, look for the `SAMPLE_LIMIT` variable. Setting it to `None` will run the full 180,000 dataset. Setting it to `1000` will run a fast test. Also, make sure to paste your local database connection link into `TUNNEL_URL`!


### 🎛️ 2.1 — Pipeline Configuration

This single cell controls the entire behavior of the ETL pipeline. 

Instead of searching through thousands of lines of code to change a setting, everything is centralized here. Let's break down what you can control:

* **Census API & Target Areas:** We are currently targeting Florida (`12`) and Hillsborough County (`057`), specifically the city limits of Tampa (`71000`). If you wanted to run this for New York City, you would simply look up the FIPS codes and change them here!
* **ACS Variables & TIGER Layers:** Defines exactly which 33 socioeconomic metrics and 11 spatial boundaries we want to extract from the government.
* **Processing Limits (`SAMPLE_LIMIT`):** Set this to a small number (like `1000`) while you are testing. When you are ready to process all the data, change it to `None`.
* **Database Connection (`TUNNEL_URL`):** Because Kaggle is a cloud server and your database is on your local machine, this URL acts as a secure tunnel connecting the two together.


In [29]:
# @title 🎛️ 2.1 — Pipeline Configuration
import os

# ── PATHS ─────────────────────────────────────────────────────────────────────
BASE_PATH          = "/kaggle/working"
DATA_DIR           = os.path.join(BASE_PATH, "data")

# ── CENSUS API ────────────────────────────────────────────────────────────────
CENSUS_API_KEY     = os.getenv("CENSUS_API_KEY", "")
CENSUS_YEAR        = "2023"
CENSUS_STATE       = "12"         # 12 = Florida (FIPS)
CENSUS_COUNTY      = "057"        # 057 = Hillsborough County (Tampa)

# ── CENSUS TIGER TARGETING ────────────────────────────────────────────────────
TARGET_STATE       = "12"         # Florida FIPS (same as CENSUS_STATE)
TARGET_COUNTY      = "057"        # Hillsborough County FIPS
TARGET_PLACE       = "71000"      # Tampa city PLACEFP code (clips TIGER to Tampa city limits)

# ── TIGER LAYERS (11 geomarketing layers — matches v1 exactly) ────────────────
TIGER_LAYERS       = ["TRACT", "BG", "PLACE", "ROADS", "AREAWATER",
                      "ZCTA5", "RAILS", "MIL", "PUMA", "EDGES", "COUSUB"]

# Converts raw Census column codes to human-readable names
TIGER_RENAME_MAP   = {
    "STATEFP":  "state_fips",         "COUNTYFP": "county_fips",
    "TRACTCE":  "tract_code",         "PLACEFP":  "place_fips",
    "PLACENS":  "place_ns",           "NAME":     "feature_name",
    "NAMELSAD": "feature_name_desc",  "ALAND":    "land_area_sqm",
    "AWATER":   "water_area_sqm",     "INTPTLAT": "latitude",
    "INTPTLON": "longitude",          "MTFCC":    "feature_class_code",
    "FUNCSTAT": "functional_status",  "GEOID":    "geoid",
    "GEOIDFQ":  "geoid_fq"
}

# ── CENSUS ACS VARIABLES ──────────────────────────────────────────────────────
# Census Bureau field codes → human-readable column names (matches v1 acs_rename_map)
ACS_VARIABLES = {
    # ── INCOME & ECONOMY ──────────────────────────────────────────────────────
    "B19013_001E": "median_household_income",
    "B19301_001E": "per_capita_income",
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",
    "B25070_010E": "high_rent_burden",
    # ── POPULATION & DEMOGRAPHICS ─────────────────────────────────────────────
    "B01001_001E": "total_population",
    "B01002_001E": "median_age",
    "B01001_020E": "senior_pop_65_plus",
    # ── RACE & ETHNICITY ──────────────────────────────────────────────────────
    "B03002_003E": "white_non_hispanic_pop",
    "B03002_004E": "black_pop",
    "B03002_006E": "asian_pop",
    "B03002_012E": "hispanic_latino_pop",
    "B05002_013E": "foreign_born_pop",
    # ── POVERTY & WELFARE ─────────────────────────────────────────────────────
    "B17001_002E": "below_poverty_line_pop",
    "B22003_002E": "snap_food_stamps_receipt",
    # ── EDUCATION & OCCUPATION ────────────────────────────────────────────────
    "B15003_022E": "bachelors_degree_pop",
    "C24050_002E": "management_business_occupations",
    "C24050_015E": "tech_finance_occupations",
    # ── HOUSING ───────────────────────────────────────────────────────────────
    "B25001_001E": "total_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25034_002E": "homes_built_after_2014",
    "B25034_010E": "homes_built_before_1939",
    # ── TRANSPORTATION & CONNECTIVITY ─────────────────────────────────────────
    "B08201_001E": "vehicles_available",
    "B08303_001E": "avg_commute_time",
    "B08301_002E": "car_commute",
    "B28002_001E": "broadband_internet_subs",
    "B28001_001E": "has_computer_smartphone",
    # ── HEALTH & FAMILY ───────────────────────────────────────────────────────
    "B27001_001E": "health_insurance_coverage",
    "B18101_001E": "pop_with_disability",
    "B11001_002E": "married_couple_families",
    "B11001_007E": "single_parent_families",
    "B13002_002E": "women_given_birth",
    # ── LANGUAGE ──────────────────────────────────────────────────────────────
    "B16001_003E": "speak_only_english",
}
# Build the comma-separated string used in the Census API URL
ACS_VARIABLES_STR = ",".join(ACS_VARIABLES.keys())

# ── GMAP DATASET ──────────────────────────────────────────────────────────────
GMAP_STATE_FILTER  = "FL"
GMAP_START_YEAR    = 2020         
SAMPLE_LIMIT       = None          # int for testing (e.g. 5000); None = full dataset

# ── NLP TRANSFORMATION ────────────────────────────────────────────────────────
EMBED_MODEL_NAME   = "BAAI/bge-large-en-v1.5"
NLP_BATCH_SIZE     = 45
NLP_CHECKPOINT_N   = 500          # Save NLP checkpoint every N batches

# ── LLM SEMANTIC ENRICHMENT ───────────────────────────────────────────────────
LLM_MODEL_NAME     = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
LLM_BATCH_SIZE     = 64
LLM_MAX_NEW_TOKENS = 256
LLM_CHECKPOINT_N   = 10          # Save LLM checkpoint every N batches

# ── OUTPUT FILES ──────────────────────────────────────────────────────────────
OUTPUT_CENSUS_DIR  = os.path.join(DATA_DIR, CENSUS_YEAR)
OUTPUT_GMAP_NLP    = os.path.join(DATA_DIR, "gmap_enriched_final.parquet")
OUTPUT_GMAP_LLM    = os.path.join(DATA_DIR, "gmap_semantic_ready.parquet")
NLP_CHECKPOINT     = os.path.join(DATA_DIR, "nlp_checkpoint.parquet")
LLM_CHECKPOINT     = os.path.join(DATA_DIR, "llm_checkpoint.parquet")
OUTPUT_ZIP_NAME    = "geoai_final_dataset"

# ── CLOUDFLARE / DATABASE ─────────────────────────────────────────────────────
# ⚠️ Paste your trycloudflare.com URL here each session
TUNNEL_URL         = "https://rendered-joining-slot-traveling.trycloudflare.com"   # <============================ CLOUDFLARE LINK
DB_USER            = os.getenv("DB_USER",     "") 
DB_PASSWORD        = os.getenv("DB_PASSWORD", "")
DB_NAME            = os.getenv("DB_NAME",     "")
DB_HOST            = "localhost"
DB_PORT            = 5433

os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Configuration loaded.")
print(f"   DATA_DIR       : {DATA_DIR}")
print(f"   SAMPLE_LIMIT   : {SAMPLE_LIMIT or 'FULL DATASET'}")
print(f"   LLM_MODEL      : {LLM_MODEL_NAME}")
print(f"   LLM_BATCH_SIZE : {LLM_BATCH_SIZE}")
print(f"   TIGER LAYERS   : {len(TIGER_LAYERS)} layers")
print(f"   ACS VARIABLES  : {len(ACS_VARIABLES)} variables")


✅ Configuration loaded.
   DATA_DIR       : /kaggle/working/data
   SAMPLE_LIMIT   : FULL DATASET
   LLM_MODEL      : unsloth/Llama-3.2-1B-Instruct-bnb-4bit
   LLM_BATCH_SIZE : 64
   TIGER LAYERS   : 11 layers
   ACS VARIABLES  : 33 variables


---
# 📚 SECTION 3 — Imports & Initializations

In Section 1, we downloaded the tools. In this section, we take those tools out of the toolbox and put them on our workbench!

This section handles all the standard Python library imports (`pandas` for dataframes, `geopandas` for maps, `os` for files) and silences unnecessary warning messages so our logs stay clean. 

More importantly, it dynamically locates and imports our custom **GeoAI Engine** (`geoai_engine.py`), which is a separate Python file we wrote that contains advanced helper functions to keep this notebook clean and easy to read.


### 📚 3.1 — All Imports

**Software Engineering Best Practice:**
In many data science notebooks, you will see `import pandas as pd` randomly written in the middle of cell 45. This is a bad practice because if you restart your notebook and try to run cell 46, it crashes because the library was imported inside a cell you skipped!

To prevent this and make our code production-ready, we follow the "Centralized Imports" rule. **Every single library this pipeline needs is imported right here.** We will never write another `import` statement for the rest of the notebook. If you restart the kernel, you only need to run this cell to have all your tools ready!


In [ ]:
# @title 📚 3.1 — All Imports (centralized — never imported again below)
import ast, re, os, sys, re, gc, json, time, shutil, socket, subprocess, urllib.request, zipfile
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import requests

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from rake_nltk import Rake

print("✅ All imports loaded.")


✅ All imports loaded.


### 🔧 3.2 — Custom GeoAI Engine Import

To keep this notebook clean and easy to read, we took all the complicated, messy background code and hid it inside a custom Python file called `geoai_engine.py` (which you can view in the project repository). 

**The Kaggle Challenge:**
When you attach a GitHub repository or a Dataset to a Kaggle notebook, Kaggle randomly places those files somewhere inside a massive folder called `/kaggle/input/`. 

This code acts like a search party. It scans the entire `/kaggle/input/` directory looking for our `geoai_engine.py` script. Once it finds it, it hooks it into the Python system path (`sys.path.insert`) so we can import it and use its advanced features later in the pipeline!


In [42]:
# @title 🔧 3.2 — Custom GeoAI Engine Import
import sys, os

found = False
for root, dirs, files in os.walk("/kaggle/input"):
    if "geoai_engine.py" in files:
        sys.path.insert(0, root)
        sys.path.insert(0, os.path.dirname(root))
        sys.path.insert(0, os.path.dirname(os.path.dirname(root)))
        print(f"  [+] Found at: {root}")
        found = True
        break

if not found:
    print("⚠️ geoai_engine.py not found in /kaggle/input")

try:
    import geoai_engine
    print(f"✅ geoai_engine loaded from: {geoai_engine.__file__}")
except ImportError as e:
    print(f"⚠️ geoai_engine not found — using fallback text normalization.")
    print(f"   Error: {e}")
    geoai_engine = None


  [+] Found at: /kaggle/input/datasets/bootcampgisatilio/bootcampgis-env/scripts/pipeline
✅ geoai_engine loaded from: /kaggle/input/datasets/bootcampgisatilio/bootcampgis-env/scripts/pipeline/geoai_engine.py


---
# 🔩 SECTION 4 — Helper Functions Library

Welcome to the Tool Shed! 🧰

In a professional ETL (Extract, Transform, Load) pipeline, we want the actual data processing steps to be clean and easy to read. If we clutter our main pipeline with hundreds of lines of code dealing with formatting text, handling arrays, or connecting to the database, it becomes unreadable.

**The Solution: Helper Functions**
In this section, we define all of our "mini-tools" (functions) exactly once. These functions do specific chores—like cleaning text, saving checkpoints, or standardizing arrays for PostgreSQL. 

By defining them all here at the top, our later ETL cells become beautifully simple. Instead of writing 50 lines of connection code later, we can just say `connect_cloudflare()`. 


### 🔩 4.1 — Helper Functions

This massive cell contains all the custom functions that power the pipeline. You don't need to change anything here, but here is a quick overview of what these functions do in the background:

* **Environment (`load_env_config`):** Scans Kaggle for your hidden `.env` file (which contains your secret database passwords) and securely loads it into memory.
* **NLP Processing (`process_text_nlp`):** Takes a raw Google Maps review, cleans out all the emojis and special characters, and uses an algorithm called **RAKE** (Rapid Automatic Keyword Extraction) to instantly pull out the top 5 most important phrases.
* **Checkpointing (`save_checkpoint`):** A safety net! If you are processing 100,000 records and Kaggle crashes at record 99,000, you don't want to start over. This function saves your progress to the hard drive every few minutes.
* **Database Connectors:** Specialized code to download the `cloudflared` software directly into Kaggle, start a background tunnel, and format complicated AI mathematical arrays into a format that PostgreSQL can understand.


In [27]:
# @title 🔩 4.1 — Helper Functions (with docstrings)

# ── ENV ───────────────────────────────────────────────────────────────────────
def find_env_file(search_root: str = "/kaggle/input") -> str | None:
    """Walk /kaggle/input to dynamically locate the .env file. Returns path or None."""
    for root, _, files in os.walk(search_root):
        if ".env" in files:
            return os.path.join(root, ".env")
    return None

def load_env_config() -> None:
    """Load .env from /kaggle/input into the process environment."""
    env_path = find_env_file()
    if env_path:
        load_dotenv(env_path, override=True)
        print(f"✅ .env loaded from: {env_path}")
    else:
        print("⚠️ .env not found in /kaggle/input.")

# ── NLP ───────────────────────────────────────────────────────────────────────
_rake = Rake()

def process_text_nlp(text: str) -> tuple[str, str]:
    """Normalize text and extract top-5 RAKE keyword phrases.

    Args:
        text: Raw review string.

    Returns:
        Tuple of (normalized_text: str, keywords_json: str).
    """
    norm = geoai_engine.normalize_text(text) if geoai_engine else text.lower().strip()
    if not norm or len(norm.strip()) < 2:
        return norm, json.dumps([])
    try:
        _rake.extract_keywords_from_text(norm)
        kws = [{"word": p, "score": round(s, 2)}
               for s, p in _rake.get_ranked_phrases_with_scores()[:5]]
        return norm, json.dumps(kws)
    except Exception:
        return norm, json.dumps([])

def is_valid_json(value) -> bool:
    """Return True if value is a valid JSON object."""
    if isinstance(value, (dict, list)):
        return True
    try:
        json.loads(str(value))
        return True
    except Exception:
        return False

# ── CHECKPOINT ────────────────────────────────────────────────────────────────
def save_checkpoint(df: pd.DataFrame, path: str) -> None:
    """Save DataFrame to Parquet as a crash-safe checkpoint.

    Args:
        df  : DataFrame to persist.
        path: Absolute path of the checkpoint file.
    """
    df.to_parquet(path, compression="snappy")

def load_checkpoint(path: str) -> pd.DataFrame | None:
    """Load a checkpoint if it exists.

    Args:
        path: Absolute path of the checkpoint file.

    Returns:
        DataFrame if checkpoint exists, else None.
    """
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  ✅ Checkpoint found: {len(df):,} records already processed.")
        return df
    return None

# ── DATABASE ──────────────────────────────────────────────────────────────────
def standardize_df_for_postgres(df: pd.DataFrame) -> pd.DataFrame:
    """Serialize numpy arrays and Python objects so psycopg2 can write them to PostgreSQL.

    Args:
        df: Input DataFrame.

    Returns:
        Copy with all numpy.ndarray columns converted to list strings.
    """
    df = df.copy()
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        if isinstance(sample, np.ndarray):
            df[col] = df[col].apply(lambda x: str(list(x)) if hasattr(x, "__iter__") else str(x))
        elif isinstance(sample, (list, dict)):
            df[col] = df[col].astype(str)
    return df

def connect_cloudflare(tunnel_url: str, local_port: int = 5433) -> None:
    """Start a Cloudflare TCP tunnel in the background.

    Args:
        tunnel_url: Full trycloudflare.com URL (https:// prefix).
        local_port: Local port to bind (default 5433).
    """
    os.system("pkill cloudflared 2>/dev/null")
    if not os.path.exists("cloudflared"):
        print("  [+] Downloading cloudflared daemon...")
        os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
        os.system("chmod +x cloudflared")
    hostname = tunnel_url.replace("https://", "").replace("http://", "")
    subprocess.Popen(f"./cloudflared access tcp --hostname {hostname} --url localhost:{local_port} &", shell=True)
    time.sleep(3)
    print(f"✅ Cloudflare tunnel active → localhost:{local_port} ↔ {hostname}")

def build_db_engine(user: str, password: str, host: str, port: int, dbname: str):
    """Build and test a SQLAlchemy engine.

    Returns:
        Connected SQLAlchemy engine.

    Raises:
        Exception if connection fails.
    """
    url = f"postgresql://{user}:{password}@{host}:{port}/{dbname}"
    eng = create_engine(url)
    with eng.connect():
        print(f"✅ Connected to {dbname} @ {host}:{port}")
    return eng

print("✅ Helper functions library loaded.")


✅ Helper functions library loaded.


---
# 📡 SECTION 5 — ETL: All Census Data (ACS + TIGER)

We are now officially entering the **"Extract"** phase of our ETL pipeline!

In this section, we connect directly to the US Government's servers to download two critical pieces of public data:
1. **The ACS (American Community Survey):** This provides rich demographic data—like median income, average age, and poverty levels—broken down into tiny neighborhood blocks called "Census Tracts".
2. **The TIGER Geometries:** This provides the physical spatial boundaries (GeoJSON polygons and lines) for roads, water, city limits, and neighborhoods.

By combining the demographic statistics from ACS with the physical map polygons from TIGER, we can build a complete socioeconomic map of the city!


### 📡 5.1 — Extract ACS Demographics

This cell is responsible for downloading our demographic statistics directly from the US Census Bureau's API. 

**How it works:**
1. **Dynamic URL Construction:** Instead of hardcoding a messy web address, this code builds the API request automatically using the exact variables (City, State, Year) you defined in **Section 2**.
2. **API Request:** It sends a secure request to the Census servers to fetch the 33 variables we asked for (like `B19013_001E`).
3. **Data Cleaning:** The raw data comes back looking like a spreadsheet of random alphanumeric codes. This script cleans out any missing values (NaNs) and uses our `ACS_VARIABLES` dictionary from Section 2 to translate those codes into human-readable column names (like `median_household_income`).
4. **GEOID Creation:** It merges the State, County, and Tract codes into a single `GEOID` column. This is a very important step, as this unique ID is what we will use later to mathematically link the statistics to the physical map boundaries!


In [ ]:
# @title 📡 5.1 — Extract ACS Demographics (Census Bureau API)

def extract_acs_demographics() -> None:
    """Download ACS 5-year demographic table for Hillsborough County census tracts.

    Applies the IPv4 auto-healing fix before the request (already applied in Cell 1.3
    but re-applied here for safety when cells are run independently).
    """
    load_env_config()

    acs_dir = os.path.join(OUTPUT_CENSUS_DIR, "acs")
    os.makedirs(acs_dir, exist_ok=True)

    # Base URL without the key
    url = (
        f"https://api.census.gov/data/{CENSUS_YEAR}/acs/acs5"
        f"?get=NAME,{ACS_VARIABLES_STR}"
        f"&for=tract:*"
        f"&in=state:{CENSUS_STATE}+county:{CENSUS_COUNTY}"
    )
    
    if getattr(sys.modules[__name__], 'CENSUS_API_KEY', None):
        url += f"&key={CENSUS_API_KEY}"
    elif os.getenv("CENSUS_API_KEY"):
        url += f"&key={os.getenv('CENSUS_API_KEY')}"

    print(f"[*] Fetching ACS {CENSUS_YEAR} — State {CENSUS_STATE}, County {CENSUS_COUNTY}...")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Prevent JSONDecodeError if API returns an empty body despite 200 OK
        if not response.text.strip():
            raise ValueError("Census API returned an empty response body.")
            
        data = response.json()
        df_acs = pd.DataFrame(data[1:], columns=data[0])
        
        # 1. Standardize column names to uppercase
        df_acs.columns = [c.upper() for c in df_acs.columns]
        
        # 2. Reconstruct GEOID correctly
        if 'STATE' in df_acs.columns and 'COUNTY' in df_acs.columns and 'TRACT' in df_acs.columns:
            df_acs['GEOID'] = df_acs['STATE'] + df_acs['COUNTY'] + df_acs['TRACT']
            
        # 3. Apply advanced engine data cleaning (removes Census special missing codes/NaNs)
        if geoai_engine:
            try:
                engine = geoai_engine.GeoAI_Engine()
                df_acs = engine.clean_acs_data(df_acs)
            except AttributeError:
                print("  ⚠️ GeoAI_Engine class not found in geoai_engine.py. Proceeding without advanced cleaning.")
                
        # 4. Rename variables to human-readable format
        df_acs = df_acs.rename(columns=ACS_VARIABLES)
        
        out = os.path.join(acs_dir, f"demographics_acs_tract_tampa_{CENSUS_YEAR}.csv")
        df_acs.to_csv(out, index=False)
        print(f"✅ ACS saved: {len(df_acs):,} tracts → {out}")
        
    except Exception as e:
        print(f"❌ ACS failed: {e}")
        # Print the raw text so you can see exactly what the API complained about
        if 'response' in locals() and hasattr(response, 'text'):
            print(f"   Raw API Response: {response.text[:200]}")


extract_acs_demographics()

### 🗺️ 5.2 — Extract TIGER Spatial Geometries

While ACS provides the *numbers* (demographics), TIGER provides the *shapes* (maps). We need 11 different layers of map data, including roads, lakes, zip codes, and neighborhood boundaries.

This cell is one of the most powerful in the pipeline. Here is the magic it performs:

1. **Dynamic Web Scraping:** It uses our `geoai_engine` to scan the US Census FTP website and automatically download the 11 ZIP files containing the raw shapefiles for the specific State and County you set in Section 2.
2. **The "Cookie Cutter" Technique (Clipping):** The Census provides maps for the entire county. But what if we only want data inside the city limits? The code looks at the `PLACE` layer, isolates the exact geometric polygon for the city of Tampa, and uses it like a cookie cutter (`gpd.clip`) to slice away all the roads and water that exist outside the city borders!
3. **Web Standardization:** It converts all the complex map coordinates into the `EPSG:4326` projection. This is the global GPS standard used by Google Maps, Mapbox, and Leaflet, ensuring our maps display correctly on the web later.

In [ ]:
# @title 🗺️ 5.2 — Extract TIGER Spatial Geometries (All Layers)

def extract_tiger_geometries() -> None:
    """Download, clip, and save all TIGER shapefiles dynamically based on config.
    
    Uses geoai_engine to automatically resolve the correct Census ZIP URLs for the 
    configured STATE and COUNTY, and clips all layers to the exact borders of TARGET_PLACE.
    """
    if not geoai_engine:
        print("❌ CRITICAL: geoai_engine is required for dynamic TIGER extraction.")
        return
        
    clipped_dir = os.path.join(OUTPUT_CENSUS_DIR, "clipped")
    os.makedirs(clipped_dir, exist_ok=True)
    
    # 1. Fetch dynamic file paths using the geoai_engine
    print(f"[*] Resolving TIGER {CENSUS_YEAR} URLs for State {TARGET_STATE}, County {TARGET_COUNTY}...")
    layers_df = pd.DataFrame({"layer_name": TIGER_LAYERS})
    
    dict_vectors = geoai_engine.download_tiger_by_inventory(
        layers_df, 
        TARGET_STATE, 
        TARGET_COUNTY, 
        year=CENSUS_YEAR
    )
    
    if 'PLACE' not in dict_vectors:
        print("❌ CRITICAL: 'PLACE' layer not found. Cannot create city clipping mask.")
        return

    # 2. Create the exact city clipping mask
    print(f"[*] Creating spatial clipping mask for Place ID {TARGET_PLACE}...")
    gdf_place = gpd.read_file(dict_vectors['PLACE'])
    city_mask = gdf_place[gdf_place['PLACEFP'] == TARGET_PLACE]
    
    if city_mask.empty:
        print(f"❌ ERROR: Place {TARGET_PLACE} not found in the downloaded PLACE layer.")
        return

    # 3. Process, Clip, and Rename each layer
    for layer_name, path in dict_vectors.items():
        print(f"  [+] Processing {layer_name}...", end=" ")
        try:
            gdf = gpd.read_file(path)
            
            # Align CRS before clipping to avoid projection errors
            if gdf.crs != city_mask.crs: 
                gdf = gdf.to_crs(city_mask.crs)
                
            # Clip to the exact city limits instead of a hardcoded bounding box
            gdf_clipped = gpd.clip(gdf, city_mask)
            
            # Convert to EPSG:4326 for standard web mapping (Leaflet, Mapbox, PostGIS)
            gdf_clipped = gdf_clipped.to_crs(epsg=4326)
            
            # Apply human-readable column names
            gdf_clipped = gdf_clipped.rename(columns=lambda x: TIGER_RENAME_MAP.get(x.upper(), x.lower()))
            
            for col in gdf_clipped.columns:
                if gdf_clipped[col].dtype == 'object':
                    # 1. Base Lowercase & Strip
                    gdf_clipped[col] = gdf_clipped[col].apply(lambda x: x.lower().strip() if isinstance(x, str) else x)
                    
                    # 2. Fix hyphen spacing in road names (e.g. "i- 4" -> "i-4")
                    if col == 'fullname':
                        gdf_clipped[col] = gdf_clipped[col].apply(lambda x: re.sub(r'\s*-\s*', '-', x) if isinstance(x, str) else x)
                    
                    # 3. Strip Census Suffixes/Prefixes from Place/Tract names
                    elif col in ['feature_name_desc', 'feature_name']:
                        gdf_clipped[col] = gdf_clipped[col].apply(
                            lambda x: re.sub(r'\s+(cdp|city|town|village)$', '', x) if isinstance(x, str) else x
                        )
                        gdf_clipped[col] = gdf_clipped[col].apply(
                            lambda x: re.sub(r'^census tract\s+', '', x) if isinstance(x, str) else x
                        )
                    else:
                        gdf_clipped[col] = gdf_clipped[col].apply(
                            lambda x: x.lower().strip() if isinstance(x, str) else x
                        )
            
            # Save to GeoJSON
            out = os.path.join(clipped_dir, f"spatial_tiger_{TARGET_PLACE}_{layer_name.lower()}.json")
            gdf_clipped.to_file(out, driver="GeoJSON")
            print(f"✅ {len(gdf_clipped):,} features clipped")
            
        except Exception as e:
            print(f"❌ Failed: {e}")

    print("\n✅ Dynamic TIGER extraction complete.")


extract_tiger_geometries()

### ✅ 5.3 — Validate Census Extraction

In an ETL pipeline, you should always verify that a step actually worked before moving on to the next one! 

This is a quick "sanity check" cell. It scans the Kaggle hard drive (`/kaggle/working/data`) to count the files we just generated. 
You should see exactly **1 ACS file** (the demographics CSV) and **11 TIGER layers** (the spatial GeoJSON files). 

If you see all 12 files listed in the output, congratulations! The US Government data extraction was a complete success, and you are ready to move on to the Google Maps data!


In [ ]:
# @title ✅ 5.3 — Validate Census Extraction
import os

acs_dir     = os.path.join(OUTPUT_CENSUS_DIR, "acs")
clipped_dir = os.path.join(OUTPUT_CENSUS_DIR, "clipped")

acs_files     = [f for f in os.listdir(acs_dir)]     if os.path.exists(acs_dir)     else []
spatial_files = [f for f in os.listdir(clipped_dir)] if os.path.exists(clipped_dir) else []

print(f"📊 Census Extraction Summary")
print(f"   ACS files     : {len(acs_files)}")
for f in acs_files: print(f"     • {f}")
print(f"   TIGER layers  : {len(spatial_files)}")
for f in spatial_files: print(f"     • {f}")
print("✅ Census data ready for PostGIS injection.")


---
# 🗺️ SECTION 6 — ETL: All GMap Data

Now that we have the official government statistics, we need human context. What do the people living in these neighborhoods actually think? For that, we turn to Google Maps!

This massive section handles the **Extraction** and **Transformation** of the Google Maps reviews. 
Because unstructured text (human reviews) is very difficult for a database to understand, we are going to use advanced Artificial Intelligence to translate human language into data points:

1. **Extraction:** We download hundreds of thousands of raw reviews using KaggleHub and filter out the old or duplicate ones.
2. **Traditional NLP (Natural Language Processing):** We use a specialized AI model to calculate the "mathematical meaning" (Vector Embeddings) of each review and extract the top keywords.
3. **Generative AI (Large Language Models):** We load a customized version of Meta's **Llama 3.2** AI directly into the Kaggle GPU. It will read every single review and extract specific JSON data points (like `price_level`, `target_audience`, and `accessibility`).

### 🗺️ 6.1 — Extract & Filter GMap Reviews

This cell connects to KaggleHub to download a massive dataset of Google Maps reviews for the state of Florida.

Because processing hundreds of thousands of reviews with AI takes a very long time, we apply three strict filters to the data before we begin:
1. **Time Filtering:** We only want recent, relevant data. We convert the Unix timestamps into readable dates and delete any review older than our `GMAP_START_YEAR` (2020).
2. **Deduplication:** We delete any duplicate reviews from the exact same user at the exact same location.
3. **Empty Text Filter:** We drop any reviews that just left a "5-star rating" but didn't write any actual text, because the AI has nothing to read!

> [!TIP]
> If you set `SAMPLE_LIMIT = 1000` in your configuration cell, it will chop the dataset down to just 1,000 reviews here so you can test the AI models quickly!

In [ ]:
# @title 🗺️ 6.1 — Extract & Filter GMap Reviews

def extract_gmap_reviews() -> pd.DataFrame:
    """Download GMap dataset via KaggleHub, filter by year, and deduplicate.
    
    Uses geoai_engine to fetch the consolidated CSV file, filters reviews 
    from GMAP_START_YEAR onwards, and cleans the dataframe for NLP.
    """
    if not geoai_engine:
        raise ImportError("geoai_engine is required to download the GMap dataset.")
        
    print("[*] Downloading GMap Florida dataset via KaggleHub...")
    input_gmap_path = geoai_engine.download_gmap_from_kaggle()
    
    if not input_gmap_path:
        raise FileNotFoundError("❌ ERROR: GMap dataset could not be downloaded or found.")
        
    print(f"  ✅ File located: {input_gmap_path}")
    df = pd.read_csv(input_gmap_path)
    print(f"  Raw rows loaded: {len(df):,}")

    # --- 1. TIME FILTERING ---
    if "time" in df.columns:
        # Let Pandas automatically infer the date format
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df = df[df["time"].dt.year >= GMAP_START_YEAR].copy()
        print(f"  After year >= {GMAP_START_YEAR}: {len(df):,}")
    else:
        print(f"  ⚠️ 'time' column missing. Skipping year filter.")

    # --- 2. DEDUPLICATION ---
    key_cols = [c for c in ["gmap_id", "user_id"] if c in df.columns]
    if key_cols:
        before = len(df)
        df = df.drop_duplicates(subset=key_cols).reset_index(drop=True)
        removed = before - len(df)
        if removed > 0:
            print(f"  After dedup: {len(df):,} (removed {removed:,} duplicates)")

    # --- 3. DROP EMPTY TEXT ---
    if "text" in df.columns:
        df = df.dropna(subset=["text"]).reset_index(drop=True)

    # --- 4. APPLY TEST LIMIT ---
    if SAMPLE_LIMIT:
        df = df.head(SAMPLE_LIMIT).copy()
        print(f"  ⚠️ TEST MODE ACTIVE: limited to {SAMPLE_LIMIT:,} records.")
    
    def normalize_gmap_categories(cat_str):
        if not isinstance(cat_str, str):
            return cat_str
        try:
            # Parse the stringified list: "['Asian grocery store']" -> ['Asian grocery store']
            cat_list = ast.literal_eval(cat_str)
            if isinstance(cat_list, list):
                # Lowercase each item and join into a clean comma-separated string
                clean_list = [c.lower().strip() for c in cat_list if isinstance(c, str)]
                return ", ".join(clean_list)
        except (ValueError, SyntaxError):
            pass 
        
        # Fallback for plain strings
        return cat_str.lower().strip()
        
    if "category" in df.columns:
        print("  [*] Normalizing GMap 'category' column...")
        df["category"] = df["category"].apply(normalize_gmap_categories)
        
    # Handle business name (stripping punctuation like apostrophes)
    name_col = "name_x" if "name_x" in df.columns else "name" if "name" in df.columns else None
    if name_col:
        print(f"  [*] Normalizing GMap '{name_col}' column...")
        df[name_col] = df[name_col].apply(
            lambda x: re.sub(r'[^\w\s]', '', x.lower().strip()) if isinstance(x, str) else x
        )

    print(f"\n✅ GMap ready: {len(df):,} records.")
    return df


df_gmap = extract_gmap_reviews()

### ✅ 6.2 — Validate GMap Extraction

Just like we did with the Census data, this is a quick "sanity check" to make sure our data looks correct before we send it into the heavy AI models. 

This cell prints out a summary showing exactly how many rows survived our filters, proves that the dates are all from 2020 or newer, and shows a quick preview of the top 3 rows. If the text looks readable and the ratings make sense, we are ready for Natural Language Processing!

In [ ]:
# @title ✅ 6.2 — Validate GMap Extraction
print(f"📊 Records : {len(df_gmap):,}")
print(f"🧬 Columns : {df_gmap.columns.tolist()}")
if "time" in df_gmap.columns:
    print(f"📅 Dates   : {df_gmap['time'].min()} → {df_gmap['time'].max()}")
print("\n👀 Sample:")
display(df_gmap[["name_x", "text", "rating", "state"]].head(3) if "name_x" in df_gmap.columns
        else df_gmap[["text", "rating"]].head(3))


### 🧬 6.3 — Transform GMap Reviews: Embeddings & Keywords

This is where the true "Artificial Intelligence" begins! Human text is messy, so we use NLP (Natural Language Processing) to translate the reviews into a format a machine can understand.

This cell does two massive transformations:
1. **Vector Embeddings (BGE-Large):** It loads a specialized AI model from the HuggingFace Hub. This model reads every review and calculates its "mathematical meaning," converting the text into an array of 1,024 numbers. Later in our database, we can use these numbers to find reviews that are semantically similar!
2. **Keyword Extraction (RAKE):** It rapidly extracts the top 5 most important phrases from each review.

**🛡️ The Checkpoint System:**
Processing 180,000 reviews can take hours. If Kaggle randomly crashes, you don't want to start over! This cell features a built-in auto-save (`save_checkpoint`). If it crashes, simply restart your kernel, run this cell again, and it will instantly resume exactly where it left off!

In [ ]:
# @title 🧬 6.3 — Transform GMap Reviews: BGE Embeddings + RAKE Keywords (Checkpointed)

def transform_reviews_nlp(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich reviews with 1024-dim BGE embeddings and RAKE keyword phrases.

    Features:
        - Processes in NLP_BATCH_SIZE batches with parallel CPU text normalization.
        - Saves a checkpoint every NLP_CHECKPOINT_N batches.
        - On restart, automatically resumes from the checkpoint.

    Args:
        df: Raw GMap DataFrame with 'text' column.

    Returns:
        DataFrame with columns: text_normalized, keywords_ai, embedding, sentiment.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[*] Loading {EMBED_MODEL_NAME} on {device.upper()}...")
    model_emb = SentenceTransformer(EMBED_MODEL_NAME, device=device)

    # ── Checkpoint resume ─────────────────────────────────────────────────────
    df_ckpt = load_checkpoint(NLP_CHECKPOINT)
    if df_ckpt is not None and "text_normalized" in df_ckpt.columns:
        already_done   = len(df_ckpt)
        df_todo        = df.iloc[already_done:].reset_index(drop=True)
        all_norm_texts = df_ckpt["text_normalized"].tolist()
        all_keywords   = df_ckpt["keywords_ai"].tolist()
        all_embeddings = [np.array(e) for e in df_ckpt["embedding"].tolist()]
        print(f"  Resuming NLP from record {already_done:,} of {len(df):,}.")
    else:
        df_todo        = df.copy().reset_index(drop=True)
        all_norm_texts, all_keywords, all_embeddings = [], [], []
        already_done   = 0
        print(f"  Fresh NLP run: {len(df):,} records.")

    total_rows    = len(df_todo)
    total_batches = (total_rows + NLP_BATCH_SIZE - 1) // NLP_BATCH_SIZE
    start_time    = time.time()
    print(f"  {total_rows:,} records remaining | {total_batches:,} batches\n")

    for i in range(0, total_rows, NLP_BATCH_SIZE):
        batch_num = (i // NLP_BATCH_SIZE) + 1
        end_idx   = min(i + NLP_BATCH_SIZE, total_rows)
        batch_df  = df_todo.iloc[i:end_idx]
        processed = i + (end_idx - i)
        elapsed   = time.time() - start_time
        eta_sec   = (elapsed / processed * (total_rows - processed)) if processed > 0 else 0

        print(f"⏳ [{batch_num:05d}/{total_batches:05d}] "
              f"Progress: {already_done+processed:,}/{len(df):,} "
              f"({(already_done+processed)/len(df)*100:.2f}%) | "
              f"Elapsed: {timedelta(seconds=int(elapsed))} | "
              f"ETA: {timedelta(seconds=int(eta_sec))}")

        with ThreadPoolExecutor(max_workers=4) as ex:
            results = list(ex.map(process_text_nlp, batch_df["text"].fillna("").tolist()))

        batch_norm, batch_kws = zip(*results)
        all_norm_texts.extend(batch_norm)
        all_keywords.extend(batch_kws)
        all_embeddings.append(
            model_emb.encode(list(batch_norm), batch_size=NLP_BATCH_SIZE,
                             show_progress_bar=False, convert_to_numpy=True)
        )

        # ── NLP checkpoint ────────────────────────────────────────────────────
        if batch_num % NLP_CHECKPOINT_N == 0:
            records_done = already_done + len(all_norm_texts)
            df_ckpt_out  = df.iloc[:records_done].copy()
            df_ckpt_out["text_normalized"] = all_norm_texts
            df_ckpt_out["keywords_ai"]     = all_keywords
            df_ckpt_out["embedding"]       = list(np.vstack(all_embeddings))
            df_ckpt_out["sentiment"]       = "neutral"
            save_checkpoint(df_ckpt_out, NLP_CHECKPOINT)
            print(f"  [NLP CHECKPOINT] {records_done:,}/{len(df):,} saved.")

    df_out = df.copy()
    df_out["text_normalized"] = all_norm_texts
    df_out["keywords_ai"]     = all_keywords
    df_out["embedding"]       = list(np.vstack(all_embeddings))
    df_out["sentiment"]       = "neutral"

    df_out.to_parquet(OUTPUT_GMAP_NLP, compression="snappy")
    if os.path.exists(NLP_CHECKPOINT):
        os.remove(NLP_CHECKPOINT)

    total_t = timedelta(seconds=int(time.time() - start_time))
    print(f"\n✅ NLP complete in {total_t} | Emb dim: {len(df_out['embedding'].iloc[0])}")
    print(f"💾 {OUTPUT_GMAP_NLP}")
    return df_out


df_gmap = transform_reviews_nlp(df_gmap)


<div style="background-color: #fff0f0; padding: 20px; border-left: 6px solid #ff4d4d; border-radius: 4px; font-size: 16px;">
  <h2 style="margin-top: 0; color: #cc0000;">⛑️ 6.4 — Mid-Pipeline Recovery Cell</h2>
  <p><b style="color: #cc0000;">DO NOT RUN THIS CELL</b> if you are running the notebook normally from top to bottom!</p>
  <p>Run this cell <b>ONLY</b> if your Kaggle session crashed or restarted after the NLP step was completed.</p>
</div>

### What is this for?
If your Kaggle session crashes, times out, or runs out of memory *after* you finished the NLP section (6.3) but *before* you finished the LLM section (6.5), this cell is your savior. 

Instead of re-running the entire notebook and waiting hours to regenerate the data, you can simply:
1. Restart your Kaggle Session.
2. Run **Cell 1.1** (to install dependencies).
3. Run this **Recovery Cell (6.4)**.

It instantly reloads your saved NLP checkpoint from the hard drive, re-imports all the required libraries, and allows you to jump straight into Cell 6.5 to finish the pipeline!


In [19]:
# @title ⛑️ 6.4 — Mid-Pipeline Recovery Cell (Run Only After Session Crash)
# ─────────────────────────────────────────────────────────────────────────────
# If your Kaggle session restarted AFTER NLP (Cell 6.3) completed but BEFORE
# the LLM (Cell 6.5) finished, run ONLY this cell and then jump directly to 6.5.
# This re-initializes the full Python environment without re-running the ETL.
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, re, gc, json, time, shutil, socket, subprocess
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm
from rake_nltk import Rake
from sentence_transformers import SentenceTransformer

# Reload config (same values as Section 2)
BASE_PATH         = "/kaggle/working"
DATA_DIR          = os.path.join(BASE_PATH, "data")
LLM_MODEL_NAME    = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
LLM_BATCH_SIZE    = 64
LLM_MAX_NEW_TOKENS= 256
LLM_CHECKPOINT_N  = 10
OUTPUT_GMAP_NLP   = os.path.join(DATA_DIR, "gmap_enriched_final.parquet")
OUTPUT_GMAP_LLM   = os.path.join(DATA_DIR, "gmap_semantic_ready.parquet")
LLM_CHECKPOINT    = os.path.join(DATA_DIR, "llm_checkpoint.parquet")

# Reload helper functions
_rake = Rake()

def is_valid_json(v):
    if isinstance(v, (dict, list)): return True
    try: json.loads(str(v)); return True
    except: return False

def save_checkpoint(df, path):
    df.to_parquet(path, compression="snappy")

def load_checkpoint(path):
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  ✅ Checkpoint: {len(df):,} records already done.")
        return df
    return None

# Load the NLP-enriched file
if os.path.exists(OUTPUT_GMAP_NLP):
    df_nlp = pd.read_parquet(OUTPUT_GMAP_NLP)
    print(f"✅ Recovery complete! df_nlp loaded: {len(df_nlp):,} records.")
    print(f"   Columns: {df_nlp.columns.tolist()}")
    print("👉 You can now run Cell 6.5 (LLM enrichment) directly.")
else:
    print(f"❌ {OUTPUT_GMAP_NLP} not found. NLP step may not have completed.")
    print("   Run from Cell 6.1 or restore session using Section 7.")


✅ Recovery complete! df_nlp loaded: 167,001 records.
   Columns: ['name_x', 'gmap_id', 'category', 'state', 'user_id', 'time', 'rating', 'text', 'text_normalized', 'keywords_ai', 'embedding', 'sentiment']
👉 You can now run Cell 6.5 (LLM enrichment) directly.


<div style="background-color: #fff9e6; padding: 20px; border-left: 6px solid #ffcc00; border-radius: 4px; font-size: 16px;">
  <h2 style="margin-top: 0; color: #b38f00;">⏱️ CRITICAL KAGGLE LIMIT: The 12-Hour GPU Timeout</h2>
  <p>If you are running the <b>full 180,000 record dataset</b> (with <code>SAMPLE_LIMIT = None</code>), the LLM inference in this cell may take around <b>14 hours</b> at default settings (approx. 19 seconds per batch of 64).</p>
  <p><b style="color: #b38f00;">Kaggle enforces a strict 12-hour maximum limit on GPU sessions.</b> If it hits 12 hours, Kaggle will forcibly kill your session, and you will lose your data!</p>
  <p>To successfully process the entire dataset, you must use one of the two strategies below:</p>
</div>

### Strategy 1: Speed Up the AI (Recommended)
You can optimize the AI to run faster and finish *under* the 12-hour limit. Go back up to **Cell 2.1 (Configuration)** and tweak these two parameters:
1. **Increase Batch Size:** Change `LLM_BATCH_SIZE = 128` (Forces the GPU to process more reviews at once. If you get a CUDA OutOfMemory error, lower it slightly).
2. **Decrease Output Length:** Change `LLM_MAX_NEW_TOKENS = 128` (Forces the AI to stop thinking earlier, as our target JSON is very short).

### Strategy 2: The "Manual Checkpoint" Save
This notebook has a built-in auto-save system (`LLM_CHECKPOINT_N = 10`). If you choose this route, you will split the processing over two days.

**Day 1: Interrupt and Save**
1. Run this cell normally and set a timer on your phone for **11 hours**.
2. When your timer goes off, **manually stop this cell** (click the square stop button). 
3. Click **"Save Version"** (top right) -> Select **"Quick Save"** -> Expand Advanced Settings and select **"Always save output"**.
4. Click Save, wait for it to finish, and close Kaggle.

**Day 2: Restore and Resume**
1. Start your Kaggle Session. 
2. Click **"+ Add Input"** (top right) -> **"Your Work"** -> Select your saved notebook version from yesterday.
3. Run **Cell 1.1** (Environment Setup) and **Cell 2.1** (Global Configuration).
4. *Skip Sections 3, 4, 5, and 6.1 through 6.3.*
5. Run **Cell 7.1** (Restore Session). This copies your partial data from yesterday into today's active folder.
6. Run **Cell 6.4** (Mid-Pipeline Recovery) to load the files into memory.
7. Run **Cell 6.5** again. It will automatically detect your checkpoint and seamlessly continue processing the rest of your dataset!

In [21]:
# @title 🧠 6.5 — Enrich Reviews with LLM Semantic Extraction (Llama 3.2 1B)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Force single GPU — prevents cross-device tensor errors

import unsloth
from unsloth import FastLanguageModel

LLM_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a data extractor. Analyze the review and extract socioeconomic and geographic indicators.
Respond ONLY with a valid JSON object.

<|eot_id|><|start_header_id|>user<|end_header_id|>

Review: {}

Expected JSON:
{{
  "price_level": "affordable/medium/expensive",
  "cuisine_ethnicity": "string or null",
  "place_type": "string",
  "target_audience": ["string"],
  "accessibility": "boolean or null",
  "geographic_ref": "street, neighborhood or null",
  "waterfront": false
}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

def enrich_with_llm(df_input: pd.DataFrame) -> pd.DataFrame:
    """Run Llama 3.2 1B batch inference to extract semantic JSON from each review.

    Features:
        - Crash-safe: saves checkpoint every LLM_CHECKPOINT_N batches.
        - Resume: detects existing checkpoint and skips already-processed records.
        - VRAM-safe: clears cache between batches.

    Args:
        df_input: NLP-enriched DataFrame with 'text' column.

    Returns:
        DataFrame with 'semantic_json' column added.
    """
    print(f"[*] Loading {LLM_MODEL_NAME}...")
    model_llm, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = LLM_MODEL_NAME,
        max_seq_length = 2048,
        load_in_4bit   = True,
        device_map     = {"": "cuda:0"},
    )
    FastLanguageModel.for_inference(model_llm)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── Checkpoint resume ─────────────────────────────────────────────────────
    df_ckpt = load_checkpoint(LLM_CHECKPOINT)
    if df_ckpt is not None and "semantic_json" in df_ckpt.columns:
        already_done = len(df_ckpt)
        df_todo      = df_input.iloc[already_done:].reset_index(drop=True)
        results      = df_ckpt["semantic_json"].tolist()
        print(f"  Resuming LLM from record {already_done:,} of {len(df_input):,}.")
    else:
        df_todo      = df_input.copy().reset_index(drop=True)
        results      = []
        already_done = 0
        print(f"  Fresh LLM run: {len(df_input):,} records.")

    print(f"  Remaining: {len(df_todo):,} | Batch size: {LLM_BATCH_SIZE}")
    start_total = time.time()

    with torch.inference_mode():
        for batch_num, i in enumerate(tqdm(range(0, len(df_todo), LLM_BATCH_SIZE), desc="LLM Inference")):
            batch_df = df_todo.iloc[i : i + LLM_BATCH_SIZE]
            prompts  = [LLM_PROMPT.format(t) for t in batch_df["text"].fillna("").tolist()]

            inputs       = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to("cuda")
            input_length = inputs["input_ids"].shape[1]
            outputs      = model_llm.generate(**inputs, max_new_tokens=LLM_MAX_NEW_TOKENS,
                                              use_cache=True, pad_token_id=tokenizer.eos_token_id)
            generated    = outputs[:, input_length:]
            decoded      = tokenizer.batch_decode(generated, skip_special_tokens=True)

            for reply in decoded:
                m = re.search(r"\{.*\}", reply, re.DOTALL)
                results.append(m.group(0) if m else "{}")

            del inputs, outputs, generated
            torch.cuda.empty_cache()

            # ── LLM checkpoint ────────────────────────────────────────────────
            if (batch_num + 1) % LLM_CHECKPOINT_N == 0:
                n_done  = len(results) # already_done + len(results)
                df_ckpt = df_input.iloc[:n_done].copy()
                df_ckpt["semantic_json"] = results
                save_checkpoint(df_ckpt, LLM_CHECKPOINT)
                elapsed = timedelta(seconds=int(time.time() - start_total))
                print(f"  [LLM CHECKPOINT] {n_done:,}/{len(df_input):,} | Elapsed: {elapsed}")

    df_out = df_input.copy()
    df_out["semantic_json"] = results
    df_out.to_parquet(OUTPUT_GMAP_LLM, compression="snappy")

    if os.path.exists(LLM_CHECKPOINT):
        os.remove(LLM_CHECKPOINT)
        print("  Checkpoint removed (pipeline complete).")

    total_t = timedelta(seconds=int(time.time() - start_total))
    print(f"\n✅ LLM enrichment complete! Time: {total_t}")
    print(f"💾 {OUTPUT_GMAP_LLM}")
    return df_out


df_nlp  = pd.read_parquet(OUTPUT_GMAP_NLP)
df_gmap = enrich_with_llm(df_nlp)

[*] Loading unsloth/Llama-3.2-1B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


  ✅ Checkpoint: 19,200 records already done.
  Resuming LLM from record 19,200 of 167,001.
  Remaining: 147,801 | Batch size: 64


LLM Inference:   0%|          | 0/2310 [00:00<?, ?it/s]

...

  [LLM CHECKPOINT] 165,760/167,001 | Elapsed: 9:58:01


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [LLM CHECKPOINT] 166,400/167,001 | Elapsed: 10:00:34


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [LLM CHECKPOINT] 167,001/167,001 | Elapsed: 10:03:41
  Checkpoint removed (pipeline complete).

✅ LLM enrichment complete! Time: 10:03:48
💾 /kaggle/working/data/gmap_semantic_ready.parquet


### ✅ 6.6 — Validate LLM Semantic Extraction

Artificial Intelligence is powerful, but it can sometimes hallucinate or break formatting rules. Because we asked the LLM to output its analysis specifically as **JSON** (JavaScript Object Notation), we need to mathematically prove that it actually listened to us. 

If the AI returned raw text paragraphs instead of JSON brackets `{ }`, our database will crash when we try to upload it!

This validation cell checks every single record to see if it is perfectly formatted JSON. It calculates a success rate (we want this to be nearly 100%) and prints out a sample of 3 records so you can verify that the AI correctly extracted the socioeconomic indicators from the messy human text.

In [22]:
# @title ✅ 6.6 — Validate LLM Semantic Extraction
df_v = pd.read_parquet(OUTPUT_GMAP_LLM)
valid     = df_v["semantic_json"].apply(is_valid_json).sum()
pct_valid = valid / len(df_v) * 100

print(f"📊 Total records   : {len(df_v):,}")
print(f"✅ Valid JSON rate  : {valid:,}/{len(df_v):,} ({pct_valid:.1f}%)")
if pct_valid < 98:
    print("⚠️ JSON valid rate below 98% — check LLM output quality.")
print("\n👀 Sample (3 records):")
for i, row in df_v.head(3).iterrows():
    print(f"\n--- Record {i+1} ---")
    print(f"Review: {str(row.get('text', ''))[:100]}...")
    try:
        print(f"JSON  : {json.dumps(json.loads(str(row['semantic_json'])), indent=2)}")
    except Exception:
        print(f"RAW   : {row['semantic_json']}")


📊 Total records   : 167,001
✅ Valid JSON rate  : 166,938/167,001 (100.0%)

👀 Sample (3 records):

--- Record 1 ---
Review: I tried this place before with a friend and I am addicted since then. I loved the good character and...
JSON  : {
  "price_level": "affordable/medium/expensive",
  "cuisine_ethnicity": "string",
  "place_type": "restaurant",
  "target_audience": [
    "young professionals",
    "foodies",
    "families"
  ],
  "accessibility": "true",
  "geographic_ref": "123 Main St, Downtown"
}

--- Record 2 ---
Review: (Translated by Google) I do not recommend it.

(Original)
No lo recomiendo....
JSON  : {
  "price_level": "affordable/medium/expensive",
  "cuisine_ethnicity": "string or null",
  "place_type": "string",
  "target_audience": [
    "string"
  ],
  "accessibility": "boolean or null",
  "geographic_ref": "street, neighborhood or null",
  "waterfront": false
}

--- Record 3 ---
Review: Food was delicious...
JSON  : {
  "price_level": "affordable",
  "cuisine_ethnicity

<div style="background-color: #e6f2ff; padding: 20px; border-left: 6px solid #0073e6; border-radius: 4px; font-size: 16px;">
  <h2 style="margin-top: 0; color: #004d99;">🔄 SECTION 7 — Session Restore (Optional)</h2>
  <p><b style="color: #004d99;">Skip this entire section if you are running the notebook normally today!</b></p>
  <p><b>The Problem it Solves:</b> Kaggle automatically deletes everything in your <code>/kaggle/working</code> folder when your session fully shuts down or times out.</p>
  <p>If you run the huge 180,000 record dataset, it will take hours. Imagine you finish the ETL pipeline, but you have to go to sleep before you can inject the data into your local PostGIS database. Before closing Kaggle, you click "Save Version" and save your output files as a Kaggle Dataset.</p>
</div>

### ✅ When to run it (Tomorrow)
Tomorrow, when you open Kaggle again, your `/kaggle/working` folder will be completely empty! But your saved dataset is mounted in the read-only `/kaggle/input` folder. 

If you uncomment and run **Cell 7.1**, it will physically copy all your processed CSVs, GeoJSONs, and Parquet files from `/kaggle/input` back into your `/kaggle/working/data` folder. This allows you to jump straight to Section 8 (Database Injection) without having to wait 14 hours for the AI to process the texts again!

### ❌ When NOT to run it (Right Now)
You are currently in an active, live session. All of your files (Census data, TIGER layers, and the LLM Parquet) are already sitting perfectly inside `/kaggle/working/data`. 

You can completely skip Section 7 today. Go directly to **Section 8 (PostGIS Injection)**, run Cell 8.1 to connect the Cloudflare tunnel, and let's get that data into your local machine!


In [18]:
# @title 🔄 7.1 — Restore Session from Saved Dataset
def restore_session(source_root: str = "/kaggle/input/notebooks/bootcampgisatilio/etl-pipeline/data", dest_root: str = DATA_DIR) -> None:

    """Copy pipeline data from /kaggle/input back to /kaggle/working after session restart.

    Args:
        source_root: Read-only Kaggle dataset mount.
        dest_root  : Writable working directory.
    """
    count = 0
    for root, _, files in os.walk(source_root):
        for fname in files:
            if fname.endswith((".parquet", ".csv", ".json", ".zip")):
                src  = os.path.join(root, fname)
                rel  = os.path.relpath(src, source_root)
                dest = os.path.join(dest_root, rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(src, dest)
                print(f"  [+] {rel}")
                count += 1
    print(f"\n✅ Restored {count} files to {dest_root}" if count else "ℹ️ Nothing to restore.")

# Uncomment to run:
# restore_session()
print("ℹ️ Session restore is disabled. Uncomment restore_session() only if session restarted.")


  [+] gmap_enriched_final.parquet
  [+] llm_checkpoint.parquet
  [+] 2023/tiger/MIL/tl_2023_us_mil.zip
  [+] 2023/tiger/AREAWATER/tl_2023_12057_areawater.zip
  [+] 2023/tiger/EDGES/tl_2023_12057_edges.zip
  [+] 2023/tiger/COUSUB/tl_2023_12_cousub.zip
  [+] 2023/tiger/PLACE/tl_2023_12_place.zip
  [+] 2023/tiger/BG/tl_2023_12_bg.zip
  [+] 2023/tiger/ROADS/tl_2023_12057_roads.zip
  [+] 2023/tiger/RAILS/tl_2023_12_rails.zip
  [+] 2023/tiger/PUMA/tl_2023_12_puma20.zip
  [+] 2023/tiger/ZCTA5/tl_2023_us_zcta520.zip
  [+] 2023/tiger/TRACT/tl_2023_12_tract.zip
  [+] 2023/clipped/spatial_tiger_71000_areawater.json
  [+] 2023/clipped/spatial_tiger_71000_puma.json
  [+] 2023/clipped/spatial_tiger_71000_tract.json
  [+] 2023/clipped/spatial_tiger_71000_edges.json
  [+] 2023/clipped/spatial_tiger_71000_cousub.json
  [+] 2023/clipped/spatial_tiger_71000_roads.json
  [+] 2023/clipped/spatial_tiger_71000_place.json
  [+] 2023/clipped/spatial_tiger_71000_bg.json
  [+] 2023/clipped/spatial_tiger_71000_zc

---
# 🏛️ SECTION 8 — Load Phase: PostGIS Injection

We have reached the final step of the ETL Pipeline: **The "Load" phase!**

Right now, all of our highly valuable, AI-processed data is sitting on a cloud server in Kaggle. If Kaggle shuts down, it's gone. We need to securely beam this data directly into our local **PostgreSQL + PostGIS** database running inside our Docker container at home.

**🚇 The Magic of the Cloudflare Tunnel:**
Because your local database is hidden behind your home internet router's firewall, Kaggle cannot normally reach it. We solve this using a **Cloudflare TCP Tunnel**. By running `cloudflared` on your local computer, it punches a secure, temporary hole through your firewall. Kaggle then connects to that tunnel to drop the data safely inside your database!

**🛑 Prerequisites Before Running:**
1. Open a terminal on your **local computer** (inside the Bootcamp folder).
2. Run this command: `.\cloudflared tunnel --url tcp://localhost:5432`
3. The terminal will give you a temporary URL (like `https://random-words.trycloudflare.com`).
4. Copy that URL and paste it into the `TUNNEL_URL` variable all the way back up in **Section 2**.
5. Once that is done, you can run cells 8.1 → 8.4 in order!


### 🔗 8.1 — Connect to Local PostGIS via Cloudflare Tunnel

This cell establishes the secure connection between Kaggle and your home computer.

**How it works:**
1. It reads the `TUNNEL_URL` you pasted in Section 2.
2. It automatically downloads the official Cloudflare software (`cloudflared`) into the Kaggle environment.
3. It spins up a background process that links Kaggle's local port `5433` directly to the Cloudflare tunnel. 
4. Finally, it uses `SQLAlchemy` (a powerful Python database library) to log in to your database using the username and password stored securely in your `.env` file. 

If this cell prints a "Connection failed" error, double-check that your terminal on your local computer is still running and that you pasted the exact URL into Section 2!

In [30]:
# @title 🔗 8.1 — Connect to Local PostGIS via Cloudflare Tunnel
load_env_config()

if TUNNEL_URL == "https://your-tunnel-name.trycloudflare.com" or not TUNNEL_URL:

    print("❌ TUNNEL_URL is not set!")
    print("   Steps:")
    print("   1. Run locally: .\\cloudflared tunnel --url tcp://localhost:5432")
    print("   2. Copy the trycloudflare.com URL")
    print("   3. Paste it as TUNNEL_URL in Section 2 (Global Configuration)")
else:
    connect_cloudflare(TUNNEL_URL, DB_PORT)
    try:
        engine = build_db_engine(DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME)
    except Exception as e:
        print(f"❌ Connection failed: {e}")


✅ .env loaded from: /kaggle/input/datasets/bootcampgisatilio/bootcampgis-env/.env


2026-09-01T23:23:15Z INF Start Websocket listener host=localhost:5433


✅ Cloudflare tunnel active → localhost:5433 ↔ rendered-joining-slot-traveling.trycloudflare.com
✅ Connected to bootcampgis @ localhost:5433


### 📦 8.2 — Load ACS Demographics + TIGER Geometries to PostGIS

Now that we are connected, it's time to start shipping the data across the internet!

This cell uses two incredibly powerful functions:
* `to_sql()` from Pandas (for our demographic CSVs)
* `to_postgis()` from GeoPandas (for our geographic GeoJSONs)

**Why this is amazing:** Normally, you would have to write hundreds of lines of `CREATE TABLE` SQL code, manually guessing whether each column was an integer, text, or a map geometry. These two functions automatically read our Kaggle files, calculate the perfect SQL data types for every column, create the 12 tables dynamically, and inject the data. 

Watch the output—you will see each table successfully created inside your local database!

In [31]:
# @title 📦 8.2 — Load ACS Demographics + TIGER Geometries to PostGIS
def load_census_to_postgis(db_engine) -> None:
    """Inject all ACS CSV and TIGER GeoJSON files into the PostGIS database.

    Args:
        db_engine: Connected SQLAlchemy engine.
    """
    acs_dir     = os.path.join(OUTPUT_CENSUS_DIR, "acs")
    clipped_dir = os.path.join(OUTPUT_CENSUS_DIR, "clipped")
    print("[*] Injecting Census data to PostGIS...")

    if os.path.exists(acs_dir):
        for f in os.listdir(acs_dir):
            if f.endswith(".csv"):
                tname = f.replace(".csv", "")
                print(f"  [+] ACS table: {tname}")
                pd.read_csv(os.path.join(acs_dir, f)).to_sql(
                    tname, con=db_engine, if_exists="replace", index=False)

    if os.path.exists(clipped_dir):
        for f in os.listdir(clipped_dir):
            if f.endswith(".json"):
                tname = f.replace(".json", "")
                print(f"  [+] TIGER table: {tname}")
                gpd.read_file(os.path.join(clipped_dir, f)).to_postgis(
                    tname, con=db_engine, if_exists="replace", index=False)

    print("✅ Census/TIGER injection complete!")


load_census_to_postgis(engine)

[*] Injecting Census data to PostGIS...
  [+] ACS table: demographics_acs_tract_tampa_2023
  [+] TIGER table: spatial_tiger_71000_mil
  [+] TIGER table: spatial_tiger_71000_rails
  [+] TIGER table: spatial_tiger_71000_bg
  [+] TIGER table: spatial_tiger_71000_zcta5
  [+] TIGER table: spatial_tiger_71000_roads
  [+] TIGER table: spatial_tiger_71000_cousub
  [+] TIGER table: spatial_tiger_71000_place
  [+] TIGER table: spatial_tiger_71000_areawater
  [+] TIGER table: spatial_tiger_71000_tract
  [+] TIGER table: spatial_tiger_71000_puma
  [+] TIGER table: spatial_tiger_71000_edges
✅ Census/TIGER injection complete!


### 🧠 8.3 — Load GMap AI-Enriched Reviews to PostGIS

Now we inject the "crown jewel" of our pipeline: the AI-processed Google Maps dataset!

Because this dataset contains advanced AI Vector Embeddings (that array of 1,024 numbers we generated in Section 6.3), we have to be very careful. PostgreSQL doesn't naturally understand Python's `numpy` arrays.

**This code does two special things:**
1. Uses our helper function `standardize_df_for_postgres` to safely package the mathematical arrays into a text string that PostgreSQL can swallow.
2. It executes a raw SQL command (`CREATE EXTENSION IF NOT EXISTS vector;`) inside your local database. This activates **pgvector**, a highly advanced plugin that turns your standard PostgreSQL database into a cutting-edge Vector Database, enabling lightning-fast similarity searches later!

In [32]:
# @title 🧠 8.3 — Load GMap AI-Enriched Reviews to PostGIS
def load_gmap_to_postgis(db_engine, input_file: str = OUTPUT_GMAP_LLM) -> None:
    """Inject the fully enriched GMap review dataset into PostGIS.

    Serializes numpy arrays and activates pgvector for similarity search.

    Args:
        db_engine  : Connected SQLAlchemy engine.
        input_file : Path to gmap_semantic_ready.parquet.
    """
    if not os.path.exists(input_file):
        print(f"⚠️ File not found: {input_file}")
        return

    df = pd.read_parquet(input_file)
    df = standardize_df_for_postgres(df)  # Convert numpy arrays → strings

    print(f"  [+] Pushing {len(df):,} records → gmap_reviews_enriched...")
    df.to_sql("gmap_reviews_enriched", con=db_engine, if_exists="replace", index=False)

    with db_engine.begin() as conn:
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
    print("✅ GMap injection complete. pgvector activated.")


load_gmap_to_postgis(engine)

  [+] Pushing 167,001 records → gmap_reviews_enriched...
✅ GMap injection complete. pgvector activated.


### 🔍 8.4 — Validate PostGIS Database

This is the moment of truth! 

We sent the data across the internet, but did it actually arrive? Instead of checking Python or looking at the Kaggle hard drive, this cell sends actual SQL queries (`SELECT table_name FROM information_schema.tables`) directly through the Cloudflare tunnel into your local database.

If it prints out a list of your 11 TIGER tables, your ACS table, and shows the correct record count for `gmap_reviews_enriched`, your ETL pipeline was an absolute, flawless success! Your data is officially safe on your home computer.

In [33]:
# @title 🔍 8.4 — Validate PostGIS Database
def validate_database(db_engine) -> None:
    """Run SQL queries to confirm all tables are present and row counts are correct.

    Args:
        db_engine: Connected SQLAlchemy engine.
    """
    try:
        tables = pd.read_sql(
            "SELECT table_name FROM information_schema.tables WHERE table_schema=\'public\'",
            con=db_engine
        )
        print(f"\n📊 Tables in database ({len(tables)}):")
        print(tables.to_string(index=False))
        count = pd.read_sql("SELECT COUNT(*) FROM gmap_reviews_enriched", con=db_engine)
        print(f"\n✅ GMap records injected: {count.iloc[0,0]:,}")
    except Exception as e:
        print(f"⚠️ Validation failed: {e}")


validate_database(engine)


📊 Tables in database (16):
                       table_name
                geography_columns
                 geometry_columns
                  spatial_ref_sys
demographics_acs_tract_tampa_2023
          spatial_tiger_71000_mil
        spatial_tiger_71000_rails
           spatial_tiger_71000_bg
        spatial_tiger_71000_zcta5
        spatial_tiger_71000_roads
       spatial_tiger_71000_cousub
        spatial_tiger_71000_place
    spatial_tiger_71000_areawater
        spatial_tiger_71000_tract
         spatial_tiger_71000_puma
        spatial_tiger_71000_edges
            gmap_reviews_enriched

✅ GMap records injected: 167,001


---
# 📦 SECTION 9 — Export Dataset

Even though we just injected the data securely into your local PostGIS database, it is always a best practice to keep a physical "hard copy" backup of your processed files!

This section takes every single file we generated over the last few hours (the ACS demographic CSVs, the 11 TIGER GeoJSON files, and the giant AI-enriched Parquet files) and compresses them all into a single, neat `.zip` file. 

**What you can do with this file:**
* Click on the "Output" folder on the right side of Kaggle and download the ZIP file directly to your desktop.
* Click "Save Version" in the top right corner of Kaggle to save the ZIP file permanently as a cloud Kaggle Dataset, meaning you will never lose this data even if your local database crashes!

### 📦 9.1 — Export Full Dataset as ZIP for Download

This simple Python script uses the built-in `shutil` (Shell Utilities) library to recursively package the entire `data/` folder and everything inside it into a single compressed archive.

Once the cell finishes running, it will print out the final filesize in Megabytes. 
To retrieve your data, look at the right-hand sidebar in Kaggle. Expand the **"Output"** section, and you will see your new `.zip` file waiting for you. You can click the three dots next to it and select **Download** to save it to your local machine!

In [34]:
# @title 📦 9.1 — Export Full Dataset as ZIP for Download
def export_dataset_zip(source_folder: str = "data", zip_name: str = OUTPUT_ZIP_NAME) -> None:
    """Compress the data folder into a downloadable ZIP archive.

    Args:
        source_folder: Relative path of folder to compress.
        zip_name     : Output archive name (without .zip extension).
    """
    out = os.path.join(BASE_PATH, zip_name)
    shutil.make_archive(out, "zip", source_folder)
    size_mb = os.path.getsize(f"{out}.zip") / 1024**2
    print(f"✅ Archive: {out}.zip ({size_mb:.1f} MB)")
    print("👉 Download: Kaggle sidebar → Output → click the .zip file")

export_dataset_zip()

✅ Archive: /kaggle/working/geoai_final_dataset.zip (4557.3 MB)
👉 Download: Kaggle sidebar → Output → click the .zip file


In [40]:
from IPython.display import FileLink
FileLink(r'geoai_final_dataset.zip')


/kaggle/working/geoai_final_dataset.zip

---
# 🔬 SECTION 10 — Automated Data Validation

Data pipelines are completely useless if the data they produce is corrupt, improperly formatted, or incomplete. 

In this section, we act as our own Quality Assurance (QA) engineers. Rather than manually clicking through database tables to ensure the data looks right, we use Python to programmatically interrogate the AI outputs and the database to mathematically prove that the ETL was successful.

### 🔬 10.1 — Automated Data Quality & Integrity Check

It is incredibly easy to make a mistake in a data pipeline. A column might get dropped, the AI might hallucinate bad JSON, or the database connection might fail silently. 

Instead of opening up the database and manually clicking through thousands of rows to verify the data, Data Engineers write **Automated Tests**. 

This cell runs an automated cross-validation script. It mathematically proves four things:
1. **Embedding Validation:** Did the HuggingFace model correctly output a 1,024-dimension mathematical vector for every review?
2. **JSON Validation:** Is the output from the Llama AI perfectly formatted JSON that a database can read?
3. **Table Count:** Did all 11 TIGER layers plus the ACS layer successfully materialize in PostGIS?
4. **Injection Match:** Does the number of rows in our Kaggle file perfectly match the number of rows that arrived in the local PostGIS database?

If you see a 4/4 passing score and a "SUCCESSFUL" banner, your ETL code is flawlessly accurate!

In [35]:
# @title 🔬 10.1 — Automated Data Quality & Integrity Check

passed, total = 0, 4

def chk(label, result, detail=""):
    global passed
    icon = "✅ PASS" if result else "❌ FAIL"
    if result: passed += 1
    print(f"  {icon} | {label}" + (f" — {detail}" if detail else ""))

print("=" * 70)
print("PIPELINE VALIDATION REPORT")
print("=" * 70)

df_nlp = pd.read_parquet(OUTPUT_GMAP_NLP) if os.path.exists(OUTPUT_GMAP_NLP) else None
df_llm = pd.read_parquet(OUTPUT_GMAP_LLM) if os.path.exists(OUTPUT_GMAP_LLM) else None

# 1. Check if Embeddings were generated correctly
if df_nlp is not None and "embedding" in df_nlp.columns:
    emb = df_nlp["embedding"].iloc[0]
    chk("Embedding dimension == 1024", len(emb) == 1024, f"got {len(emb)}")
else:
    chk("Embedding dimension == 1024", False, "missing")

# 2. Check if the AI successfully generated valid JSON format
if df_llm is not None and "semantic_json" in df_llm.columns:
    rate = df_llm["semantic_json"].apply(is_valid_json).mean()
    chk("LLM JSON valid rate >= 98%", rate >= 0.98, f"{rate*100:.1f}%")
else:
    chk("LLM JSON valid rate >= 98%", False, "missing")

# 3 & 4. Check if the database received the data correctly
try:
    t = pd.read_sql("SELECT COUNT(*) as n FROM information_schema.tables WHERE table_schema=\'public\'", con=engine)
    chk("PostGIS tables >= 12", t.iloc[0,0] >= 12, f"{t.iloc[0,0]} found")
    
    c = pd.read_sql("SELECT COUNT(*) FROM gmap_reviews_enriched", con=engine)

    exp = len(df_nlp) if df_nlp is not None else 0
    chk("PostGIS row count matches file", c.iloc[0,0] == exp, f"DB={c.iloc[0,0]:,} | file={exp:,}")
except Exception as e:
    chk("PostGIS tables >= 12", False, str(e))
    chk("PostGIS row count matches file", False, "DB unreachable")

print("=" * 70)
print(f"RESULT: {passed}/{total} checks passed.")
print("🎉 PIPELINE EXECUTION SUCCESSFUL!" if passed == total else "⚠️ Some checks failed. Review above.")
print("=" * 70)


PIPELINE VALIDATION REPORT
  ✅ PASS | Embedding dimension == 1024 — got 1024
  ✅ PASS | LLM JSON valid rate >= 98% — 100.0%
  ✅ PASS | PostGIS tables >= 12 — 16 found
  ✅ PASS | PostGIS row count matches file — DB=167,001 | file=167,001
RESULT: 4/4 checks passed.
🎉 PIPELINE EXECUTION SUCCESSFUL!


---
# 🛡️ SECTION 11 — Quality Assurance & Testing

This section serves as the final QA (Quality Assurance) checkpoint for the ETL pipeline. It proves programmatically that the data logic is sound and that the generated output files meet production standards.

**🧪 11.1 Unit Tests:**
Tests the **internal logic** of the custom Python functions (like `is_valid_json` and `process_text_nlp`) using fake, temporary data to ensure they don't crash or return incorrect data types.

**📁 11.2 Integration Tests:**
Tests the **physical files** written to the Kaggle hard drive. It guarantees that:
* The Parquet and GeoJSON files actually exist.
* The AI didn't accidentally drop required columns (like `sentiment` or `embedding`).
* The vector embeddings are exactly 1,024 dimensions long.
* The Census API successfully downloaded all required spatial layers.

**🚦 When to run this section:**
* **At the very end of your pipeline execution:** It serves as the final stamp of approval that your ETL succeeded and the data is production-ready.
* **After modifying code:** If you ever change the functions in `geoai_engine.py` or switch the AI model, you run these tests to guarantee your changes didn't accidentally break the pipeline.

**🛑 When NOT to run this section:**
* **During partial runs:** If you are only testing the Census download (Section 5) and haven't run the LLM (Section 6), these tests will fail because the GMap Parquet files won't exist yet. Run this only when the *entire* pipeline is complete.

### 🧪 11.1 — Unit Tests (Pure Functions)

In Software Engineering, a **Unit Test** is a piece of code that tests a very specific, isolated piece of logic to prove it works perfectly under different conditions.

This cell does not touch our 180,000 Kaggle dataset. Instead, it feeds fake, temporary data into our helper functions (from Section 4) to ensure they respond correctly. 

**What it tests:**
1. **JSON Parser:** It feeds a valid JSON string, a valid Python dictionary, and a broken garbage string (`"not json at all!!"`) into our JSON verifier to make sure it only accepts the correct formats.
2. **NLP Engine:** It feeds the sentence `"Great food at the downtown restaurant!"` into the NLP engine and verifies that the engine actually spits out a 2-part tuple containing normalized text and valid JSON keywords.
3. **Database Formatting:** It feeds a fake array of math numbers into the `standardize_df_for_postgres` function to ensure it converts them to text strings.
4. **Checkpoint System:** It creates a tiny fake spreadsheet, saves it to the hard drive, loads it back, checks that no rows were lost, and deletes the test file.

In [43]:
# @title 🧪 11.1 — Unit Tests (Pure Functions)
# Tests individual helper functions without any data dependencies.

import traceback
import os
import pandas as pd
import numpy as np

_unit_passed = 0
_unit_total  = 0

def unit_test(name, fn):
    global _unit_passed, _unit_total
    _unit_total += 1
    try:
        fn()
        print(f"  ✅ {name}")
        _unit_passed += 1
    except AssertionError as e:
        print(f"  ❌ {name} — AssertionError: {e}")
    except Exception as e:
        print(f"  ❌ {name} — {type(e).__name__}: {e}")

print("=" * 55)
print("UNIT TESTS")
print("=" * 55)

# 1. is_valid_json — valid dict
def _test_json_dict():
    assert is_valid_json({"a": 1}) == True
unit_test("is_valid_json accepts dict", _test_json_dict)

# 2. is_valid_json — valid JSON string
def _test_json_string():
    assert is_valid_json('{"price": "affordable"}') == True
unit_test("is_valid_json accepts JSON string", _test_json_string)

# 3. is_valid_json — invalid string
def _test_json_garbage():
    assert is_valid_json("not json at all!!") == False
unit_test("is_valid_json rejects garbage string", _test_json_garbage)

# 4. process_text_nlp
def _test_process_text():
    r = process_text_nlp("Great food at the downtown restaurant!")
    assert isinstance(r, tuple), "Expected tuple"
    assert len(r) == 2, "Expected length 2"
    assert isinstance(r[0], str), "Expected string for norm text"
    assert isinstance(r[1], str), "Expected string for keywords"
    assert is_valid_json(r[1]), "Keywords should be JSON"
unit_test("process_text_nlp returns valid tuple & json", _test_process_text)

# 5. standardize_df_for_postgres
def _test_standardize():
    df_test = pd.DataFrame({"emb": [np.array([1.0, 2.0]), np.array([4.0, 5.0])]})
    # This might error if standardize_df_for_postgres wasn't defined manually in Kaggle
    try:
        result = standardize_df_for_postgres(df_test)
        assert isinstance(result["emb"].iloc[0], str), "embedding should be string"
    except NameError:
        print("  ⚠️ Skipped: standardize_df_for_postgres not found in global scope")
unit_test("standardize_df_for_postgres converts numpy to str", _test_standardize)

# 6. save_checkpoint + load_checkpoint round-trip
def _test_checkpoint():
    tmp = os.path.join(DATA_DIR, "_test_checkpoint.parquet")
    df_in = pd.DataFrame({"a": [1, 2, 3], "b": ["x", "y", "z"]})
    save_checkpoint(df_in, tmp)
    df_out = load_checkpoint(tmp)
    assert df_out is not None, "Loaded checkpoint is None"
    assert len(df_out) == 3, "Length mismatch"
    os.remove(tmp)
unit_test("save_checkpoint + load_checkpoint round-trip", _test_checkpoint)

print(f"\nUnit Tests: {_unit_passed}/{_unit_total} passed.")

UNIT TESTS
  ✅ is_valid_json accepts dict
  ✅ is_valid_json accepts JSON string
  ✅ is_valid_json rejects garbage string
  ✅ process_text_nlp returns valid tuple & json
  ✅ standardize_df_for_postgres converts numpy to str
  ✅ Checkpoint found: 3 records already processed.
  ✅ save_checkpoint + load_checkpoint round-trip

Unit Tests: 6/6 passed.


### 🧪 11.2 — Integration Tests (File & Schema Validation)

In Software Engineering, an **Integration Test** checks if the different moving parts of your pipeline successfully connected and produced real-world results. 

While the Unit Tests (in 11.1) used fake data to test logic, this cell looks at the actual `/kaggle/working/data` folder on the hard drive to verify the final deliverables. 

**What it tests:**
1. **Physical Files:** Does `gmap_semantic_ready.parquet` physically exist? Did the Census API actually save all the TIGER `.json` map files?
2. **Schema Integrity:** Did the AI accidentally delete the `sentiment` or `embedding` columns? It checks the parquet file to ensure all 5 required columns are present.
3. **Data Quality Thresholds:** It opens the final LLM file and mathematically proves that at least 98% of the AI's output is perfectly valid JSON. If it falls below 98%, it fails the test, warning you that the AI model hallucinated too much.

If this cell gives you an 8/8 passing score, your dataset is structurally perfect and ready to be loaded into the database or exported!

In [37]:
# @title 🧪 11.2 — Integration Tests (File & Schema Validation)
# Tests that output files exist, have the right schema, and meet quality thresholds.

import os
import pandas as pd

_int_passed = 0
_int_total  = 0

def int_test(name, fn):
    global _int_passed, _int_total
    _int_total += 1
    try:
        fn()
        print(f"  ✅ {name}")
        _int_passed += 1
    except AssertionError as e:
        print(f"  ❌ {name} — {e}")
    except Exception as e:
        print(f"  ❌ {name} — {type(e).__name__}: {e}")

print("=" * 55)
print("INTEGRATION TESTS")
print("=" * 55)

# 1. NLP parquet file exists
def _test_nlp_file():
    assert os.path.exists(OUTPUT_GMAP_NLP), f"Missing: {OUTPUT_GMAP_NLP}"
int_test("NLP parquet file exists", _test_nlp_file)

# 2. NLP parquet has required columns
def _test_nlp_schema():
    df = pd.read_parquet(OUTPUT_GMAP_NLP)
    required = {"text", "text_normalized", "keywords_ai", "embedding", "sentiment"}
    missing = required - set(df.columns)
    assert not missing, f"Missing columns: {missing}"
int_test("NLP parquet has required columns", _test_nlp_schema)

# 3. Embedding dimension is 1024
def _test_emb_dim():
    df = pd.read_parquet(OUTPUT_GMAP_NLP)
    emb = df["embedding"].iloc[0]
    assert len(emb) == 1024, f"Expected 1024, got {len(emb)}"
int_test("Embedding dimension == 1024 (BGE-Large)", _test_emb_dim)

# 4. NLP row count > 0
def _test_nlp_rows():
    df = pd.read_parquet(OUTPUT_GMAP_NLP)
    assert len(df) > 0, "NLP output is empty"
int_test("NLP output has rows > 0", _test_nlp_rows)

# 5. LLM parquet file exists
def _test_llm_file():
    assert os.path.exists(OUTPUT_GMAP_LLM), f"Missing: {OUTPUT_GMAP_LLM}"
int_test("LLM parquet file exists", _test_llm_file)

# 6. LLM JSON valid rate >= 98%
def _test_json_rate():
    df     = pd.read_parquet(OUTPUT_GMAP_LLM)
    rate   = df["semantic_json"].apply(is_valid_json).mean()
    assert rate >= 0.98, f"JSON valid rate {rate*100:.1f}% is below 98%"
int_test("LLM JSON valid rate >= 98%", _test_json_rate)

# 7. Census ACS file exists
def _test_acs_file():
    acs_dir = os.path.join(OUTPUT_CENSUS_DIR, "acs")
    assert os.path.exists(acs_dir), "ACS directory not found"
    assert any(f.endswith(".csv") for f in os.listdir(acs_dir)), "No ACS CSV found"
int_test("ACS CSV file exists", _test_acs_file)

# 8. TIGER spatial files exist
def _test_tiger_files():
    clip_dir = os.path.join(OUTPUT_CENSUS_DIR, "clipped")
    assert os.path.exists(clip_dir), "Clipped directory not found"
    count = len([f for f in os.listdir(clip_dir) if f.endswith(".json")])
    assert count >= 5, f"Less than 5 TIGER layers found (got {count})"
int_test("TIGER GeoJSON files exist (>= 5)", _test_tiger_files)

print(f"\nIntegration Tests: {_int_passed}/{_int_total} passed.")


INTEGRATION TESTS
  ✅ NLP parquet file exists
  ✅ NLP parquet has required columns
  ✅ Embedding dimension == 1024 (BGE-Large)
  ✅ NLP output has rows > 0
  ✅ LLM parquet file exists
  ✅ LLM JSON valid rate >= 98%
  ✅ ACS CSV file exists
  ✅ TIGER GeoJSON files exist (>= 5)

Integration Tests: 8/8 passed.


### 🧪 11.3 — End-to-End Smoke Test

In Software Engineering, a **"Smoke Test"** gets its name from plumbing: if you turn on the pipes and no smoke comes out of the cracks, the system is fundamentally sealed and working!

Instead of running a single unit test on fake data, or checking if the final file exists, a Smoke Test takes a tiny sample of real data (just 10 records) and runs it through the *entire* lifecycle of the pipeline at lightning speed. 

It validates the schema, the mathematical embeddings, the JSON formatting, the checkpoint saving system, and the database standardizer all in one go. If this passes, you can confidently run the pipeline on 180,000 records knowing that it won't crash halfway through!

In [38]:
# @title 🧪 11.3 — End-to-End Smoke Test (10 Records)
# Runs the COMPLETE pipeline on 10 records to verify all steps produce valid outputs.
# Does NOT modify the production output files.

import tempfile, os
print("=" * 55)
print("END-TO-END SMOKE TEST (10 records)")
print("=" * 55)

SMOKE_OUTPUT_NLP = "/kaggle/working/data/_smoke_nlp.parquet"
SMOKE_OUTPUT_LLM = "/kaggle/working/data/_smoke_llm.parquet"
SMOKE_CKPT       = "/kaggle/working/data/_smoke_ckpt.parquet"

try:
    # 1. Load 10 records from the real NLP output
    df_smoke = pd.read_parquet(OUTPUT_GMAP_NLP).head(10).copy()
    print(f"  ✅ Loaded 10 records from {OUTPUT_GMAP_NLP}")

    # 2. Validate schema
    required = {"text", "text_normalized", "keywords_ai", "embedding", "sentiment"}
    missing  = required - set(df_smoke.columns)
    assert not missing, f"Missing columns: {missing}"
    print(f"  ✅ Schema check passed: {list(df_smoke.columns)}")

    # 3. Validate embedding dimensions
    emb_dim = len(df_smoke["embedding"].iloc[0])
    assert emb_dim == 1024, f"Expected 1024 dims, got {emb_dim}"
    print(f"  ✅ Embedding dimension: {emb_dim}")

    # 4. Validate keywords are valid JSON
    kw_valid = df_smoke["keywords_ai"].apply(is_valid_json).mean()
    assert kw_valid == 1.0, f"Some keywords are invalid JSON: {kw_valid}"
    print(f"  ✅ Keyword JSON rate: {kw_valid*100:.0f}%")

    # 5. Simulate checkpoint save/load
    save_checkpoint(df_smoke, SMOKE_CKPT)
    df_loaded = load_checkpoint(SMOKE_CKPT)
    assert df_loaded is not None and len(df_loaded) == 10
    os.remove(SMOKE_CKPT)
    print(f"  ✅ Checkpoint round-trip: 10 records saved and reloaded correctly")

    # 6. Validate standardize_df_for_postgres
    df_std = standardize_df_for_postgres(df_smoke)
    assert isinstance(df_std["embedding"].iloc[0], str), "Embedding not serialized"
    print(f"  ✅ standardize_df_for_postgres: numpy arrays converted to strings")

    print("\n✅ SMOKE TEST PASSED — All 6 steps verified on 10 records.")

except AssertionError as e:
    print(f"\n❌ SMOKE TEST FAILED: {e}")
except Exception as e:
    import traceback
    print(f"\n❌ SMOKE TEST ERROR: {e}")
    traceback.print_exc()


END-TO-END SMOKE TEST (10 records)
  ✅ Loaded 10 records from /kaggle/working/data/gmap_enriched_final.parquet
  ✅ Schema check passed: ['name_x', 'gmap_id', 'category', 'state', 'user_id', 'time', 'rating', 'text', 'text_normalized', 'keywords_ai', 'embedding', 'sentiment']
  ✅ Embedding dimension: 1024
  ✅ Keyword JSON rate: 100%
  ✅ Checkpoint found: 10 records already processed.
  ✅ Checkpoint round-trip: 10 records saved and reloaded correctly
  ✅ standardize_df_for_postgres: numpy arrays converted to strings

✅ SMOKE TEST PASSED — All 6 steps verified on 10 records.
